# Nowcasting Meteosat imagery over Ireland with a fine-tuned transformer

This project forecasts what the new **Meteosat Third Generation** satellite will see over Ireland one
and two hours from now.

It starts from NVIDIA's [StormScope-Meteosat](https://huggingface.co/nvidia/stormscope-meteosat), a
diffusion **transformer** pre-trained to predict the next satellite image 10 minutes ahead over
Europe. We keep its 613 million weights frozen and train small **LoRA** adapters (about 0.1% of the
weights) so that it works on **hourly time series** of images over Ireland. The forecasts are then
compared with two baselines: **persistence** (the next hour looks like the last one) and the
**untouched pre-trained model**.

All the code lives in the `earth_forecasting` package
([GitHub](https://github.com/mich3leRusso/earth-forecasting)); this notebook only calls it and shows
the results. The trained adapter is on the Hugging Face Hub at
[`michele007rok/StormScope_Ireland`](https://huggingface.co/michele007rok/StormScope_Ireland).

**Running it.** The first cell makes the notebook self-contained: if the project isn't already
installed, it clones the GitHub repository, installs the pinned versions in `requirements.txt` with
pip into this notebook's Python, and moves into the repository folder. It works on a laptop, on a cluster
or on Google Colab; you need an **NVIDIA GPU** (8 GB is enough) and a free
[EUMETSAT](https://api.eumetsat.int/api-key/) API key for the satellite images.

**`TRAIN`**, in the settings cell after the setup (where the EUMETSAT key goes too), chooses what runs:
- `False` (default): the adapter trained for this project is loaded from the Hugging Face Hub, and
  the training curves and test scores are read from the repository's `results/` folder. Only the
  eight images of the example are downloaded: a few minutes in total.
- `True`: everything is redone here. The training and validation months are downloaded (about
  2,900 images, several hours), the adapter is trained for 12 epochs (about 3 hours on an 8 GB GPU)
  and scored on 1 – 15 October (about 2 hours); the rest of the notebook then shows the new run.

In [ ]:
# Setup: makes the notebook portable (laptop, cluster, Google Colab). Finds or clones the repository,
# makes this notebook's Python match requirements.txt exactly (with pip), and runs from the repository folder.
import importlib, os, re, subprocess, sys
from importlib import metadata
from pathlib import Path

REPO_URL = "https://github.com/mich3leRusso/earth-forecasting.git"

root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src/earth_forecasting").is_dir()), None)
if root is None:
    root = Path.cwd() / "earth-forecasting"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
os.chdir(root)


def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def applies(marker):  # e.g. "python_full_version < '3.12'": some packages are pinned per Python version
    if not marker.strip():
        return True
    try:
        from packaging.markers import Marker
    except ImportError:
        from pip._vendor.packaging.markers import Marker
    return Marker(marker).evaluate()


# pinned "name==version" lines that don't match what this Python has (an older pandas, xarray, ... breaks the code)
pins = [re.match(r"([A-Za-z0-9_.\-]+)==([^\s;]+)\s*(?:;(.*))?", r) for r in Path("requirements.txt").read_text().splitlines()]
wrong = [m.group(1) for m in pins if m and applies(m.group(3) or "") and installed(m.group(1)) != m.group(2)]
if wrong or installed("earth-forecasting") is None:
    print(f"installing {len(wrong)} pinned packages ({', '.join(wrong[:6])}{', ...' if len(wrong) > 6 else ''}) and the project")
    before = {m: installed(m) for m in ("numpy", "pandas", "xarray", "matplotlib", "torch", "scipy") if m in sys.modules}
    if subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True).returncode:
        # this Python has no pip (e.g. a venv made by uv): bootstrap it from the standard library
        if subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], capture_output=True).returncode:
            raise SystemExit(f"{sys.executable} has no pip and can't install it; pick another kernel")
    pip = [sys.executable, "-m", "pip", "install", "-q"]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": ""}  # GPU hidden: NATTEN takes its PyTorch backend instead of compiling CUDA code
    requirements = "requirements.txt"
    # the pinned PyTorch is built for CUDA 13 (NVIDIA driver 580+); with an older driver use its CUDA 12.6 build
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout if subprocess.run(["which", "nvidia-smi"], capture_output=True).returncode == 0 else ""
    cuda = re.search(r"CUDA Version: (\d+)\.", smi)
    if cuda and int(cuda.group(1)) < 13:
        lines = Path("requirements.txt").read_text().splitlines()
        torch_pins = [r for r in lines if r.startswith(("torch==", "torchvision=="))]
        subprocess.run([*pip, *torch_pins, "--index-url", "https://download.pytorch.org/whl/cu126"], check=True, env=env)
        cuda13 = lambda r: r.startswith(("torch==", "torchvision==", "cuda-")) or (r.startswith("nvidia-") and not r.startswith("nvidia-physicsnemo"))
        Path("requirements-cu126.txt").write_text("\n".join(r for r in lines if not cuda13(r)) + "\n")
        requirements = "requirements-cu126.txt"
    subprocess.run([*pip, "-r", requirements], check=True, env=env)
    subprocess.run([*pip, "--no-deps", "-e", "."], check=True)

    changed = [m for m, v in before.items() if installed(m) != v]
    if changed:  # already loaded in their old version: only a fresh Python sees the new ones
        raise SystemExit(f"Done. {', '.join(changed)} changed version: restart the kernel "
                         "(Colab: Runtime > Restart session) and run this cell again.")
    sys.path.insert(0, str(root / "src"))  # the editable install is only seen by new Python processes
    importlib.invalidate_caches()

import earth_forecasting, torch
assert torch.cuda.is_available(), "StormScope needs an NVIDIA GPU (on Colab: Runtime > Change runtime type > GPU)"
print(f"repository: {root}\ntorch {torch.__version__} on {torch.cuda.get_device_name()}")

### ⚠️ Settings: put your EUMETSAT API key here

The satellite images are downloaded from EUMETSAT, which needs the **consumer key** and
**consumer secret** of a free account: log in at
[api.eumetsat.int/api-key](https://api.eumetsat.int/api-key/) and copy them into the cell below.
If you leave them empty, the notebook asks for them (hidden) when it needs them. Don't share the
notebook with your keys written in it.

In [ ]:
EUMETSAT_CONSUMER_KEY = ""     # <- your EUMETSAT consumer key
EUMETSAT_CONSUMER_SECRET = ""  # <- your EUMETSAT consumer secret

TRAIN = False  # False: adapter from the Hugging Face Hub + saved results. True: download, train and score here (hours)

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from physicsnemo.experimental.peft import is_lora_layer

from earth_forecasting import data, evaluate, plots, train
from earth_forecasting.config import load_config, region_of
from earth_forecasting.data import FrameWindows, download_frames
from earth_forecasting.evaluate import finetuned_model

train_cfg = load_config("train.yaml", args=[])  # how the adapter was trained
cfg = load_config("evaluate.yaml", args=[])     # how it is tested: unseen dates, adapter from the Hub
step = np.timedelta64(cfg.data.step_minutes, "m")
RESULTS = Path("results")  # training curves and test scores of the published adapter

# EUMETSAT key: from the settings cell, else the environment, else src/earth_forecasting/constants.py, else asked
if EUMETSAT_CONSUMER_KEY:
    os.environ["EUMETSAT_CONSUMER_KEY"], os.environ["EUMETSAT_CONSUMER_SECRET"] = EUMETSAT_CONSUMER_KEY, EUMETSAT_CONSUMER_SECRET
try:
    from earth_forecasting.constants import EUMETSAT_KEY as saved_key  # the repository ships this file empty
except ImportError:
    saved_key = ""
if not (os.environ.get("EUMETSAT_CONSUMER_KEY") or saved_key):
    os.environ["EUMETSAT_CONSUMER_KEY"] = getpass("EUMETSAT consumer key: ")
    os.environ["EUMETSAT_CONSUMER_SECRET"] = getpass("EUMETSAT consumer secret: ")

if TRAIN:
    cfg.adapter = None                                     # score the adapter trained here, not the Hub one
    cfg.output.checkpoints = train_cfg.output.checkpoints  # its best epoch is saved there
    data.run(train_cfg)   # training + validation months
    train.run(train_cfg)  # LoRA training; curves logged to TensorBoard
    data.run(cfg)         # test period
    evaluate.run(cfg)     # fine-tuned vs base vs persistence -> runs/eval_test/rmse_test.csv
    RESULTS = Path(cfg.output.tensorboard)

## 1. Data: hourly satellite images

The data are **Level 1c images from the Flexible Combined Imager (FCI)** on Meteosat-12 (MTG-I1),
downloaded from the [EUMETSAT Data Store](https://data.eumetsat.int). Each image has **16 channels**:
5 visible, 3 near-infrared, 2 water-vapour and 6 thermal-infrared bands. That lets the model see
clouds in daylight and their temperature day and night.

We crop the full disk to Ireland, which gives 128 × 224 pixels at the satellite's 2 km grid (about
3–4 km on the ground at this latitude), and keep one image per hour.

**One sample** is a short time series: six consecutive hourly images (the last five hours and now)
plus the image one hour later, which is the target. The periods are split in time, never mixed:

| | Forecast start times | Purpose |
|---|---|---|
| Training | 1 June – 31 August 2026 | fit the adapters |
| Validation | 2 – 8 September 2026 | choose the best epoch |
| Test | 1 – 15 October 2026 | final scores, seen by nothing else |

NVIDIA trained the base model on November 2024 – May 2026, so every date here is new to it.
A handful of hours are missing or incomplete on EUMETSAT's side, and the windows that touch them are
skipped: 2,185 training windows, 168 for validation and 360 for testing.

The cell below downloads only what the example needs: the eight hourly images around
12 October 2026, 12:00 UTC (about 45 MB of transfer each, a few minutes in total). Images already on
disk are not downloaded again.

In [ ]:
now = np.datetime64("2026-10-12T12:00")  # forecast start: the six images up to now, then +1h and +2h
download_frames(now - 5 * step, now + (cfg.eval.lead_steps + 1) * step, cfg.data.dir, region_of(cfg), step)
example = FrameWindows(now, now + step, cfg.data.dir, step, lead_steps=cfg.eval.lead_steps)[0]

model, adapter = finetuned_model(cfg)  # StormScope (2.4 GB, downloaded on first use) + the adapter
plots.window_figure(model, *example);

## 2. Model: a pre-trained diffusion transformer, fine-tuned with LoRA

Each of StormScope's two "experts" is a **Diffusion Transformer (DiT)**: the image is cut into
4 × 4-pixel patches, which become tokens, and 16 transformer blocks process them with
neighborhood attention and MLPs. A forecast starts from noise and is refined over 48 denoising
steps, each conditioned on the six past images, the sun angles and the terrain. One expert handles
the noisy early steps (large-scale structure), the other the fine detail at the end.

The original model predicts 10 minutes ahead. We switch it to **one-hour steps**: it reads six
hourly images and predicts the next hour. To go further, it feeds its own prediction back in.
The pre-trained weights stay frozen; we train **LoRA adapters** (rank 8) on the attention
projections of the last 8 blocks of both experts, plus the final layer that turns tokens back into
the 16 channels.

In [ ]:
lora = sum(p.numel() for m in model.modules() if is_lora_layer(m) for n, p in m.named_parameters(recurse=False) if "lora" in n)
total = sum(p.numel() for p in model.parameters())
print(f"adapter: {adapter}")
print(f"trained: {lora:,} of {total:,} weights ({lora / total:.2%})")

## 3. Training

Training runs on a laptop GPU with 8 GB of memory, so the batch size is 1, with gradients
accumulated over 8 samples. Each sample trains one expert on the standard diffusion
(denoising) loss. We use the Muon optimizer with a linear warmup and a cosine learning-rate decay,
for 12 epochs over the 2,185 training windows: about 14 minutes per epoch.

Epoch 0 is the untouched pre-trained model on the validation week, which gives the starting point.

In [ ]:
if TRAIN:
    curves = plots.training_curves(train_cfg.output.tensorboard)  # the run just made
else:
    curves = pd.read_csv(RESULTS / "training_curves.csv", index_col="epoch")  # the published adapter's run
ax = curves.plot(marker="o", xlabel="epoch", ylabel="denoising loss", title="Training and validation loss", grid=True)
curves.round(4)

## 4. Evaluation on unseen dates (October 2026)

For 60 forecast start times spread over 1 – 15 October, each model forecasts one and two hours
ahead, and every forecast is compared with what the satellite actually observed. The score is the
**RMSE** over all land and sea pixels, computed per channel on the normalized scale so that the 16
channels are comparable. Lower is better.

Both models use the same diffusion noise for each date, so the difference between them comes from
the adapters alone. The last two columns show how much each model improves on persistence.

With `TRAIN = False` the scores are read from `results/rmse_test.csv`; with `TRAIN = True` they
were just computed for the new adapter.

In [ ]:
mean, per_channel = plots.rmse_summary(RESULTS / "rmse_test.csv")
mean[plots.MODELS].plot.bar(rot=0, xlabel="lead time (hours)", ylabel="mean RMSE (normalized)", title="Forecast error, 1 - 15 October")
mean

In [ ]:
per_channel.round(3)

## 5. One forecast, side by side

The same start time as the example above, for a thermal-infrared channel (cloud-top temperature,
day and night) and a visible one (reflected sunlight). Each row is a lead time. From left to right:
persistence, the pre-trained model, the fine-tuned model, and what the satellite actually saw.

In [ ]:
plots.forecast_figure(model, *example, lead_steps=cfg.eval.lead_steps, channel="fci105ir");

In [ ]:
plots.forecast_figure(model, *example, lead_steps=cfg.eval.lead_steps, channel="fci06vis");

## 6. What we learned

- **It works on unseen dates.** Over 1 – 15 October the fine-tuned model beats persistence by
  **20% at +1h and 28% at +2h**, and the untouched pre-trained model by **3% and 12%**.
- **The gain grows with lead time.** The base model was trained for 10-minute steps; fed hourly
  frames and then its own +1h prediction, it drifts faster. The adapters learned hourly dynamics,
  so at +2h the gap to the base model is four times larger than at +1h.
- **Most of the gain is in reflected sunlight.** Visible and near-infrared channels improve most
  (e.g. `fci04vis` at +2h: 0.348 vs 0.555, about 37% lower). The thermal-infrared channels get
  slightly worse than the base model, by 1–4%, though still well ahead of persistence.
- **Training only 0.1% of the weights was enough**, on a laptop GPU, in under three hours.

**Limits.** The scores come from 60 start times and a single diffusion sample each, so differences
of 1–2% are within the noise. A diffusion model is probabilistic: an ensemble of samples, scored
with CRPS, would judge it more fairly than the RMSE of one sample. The model only sees the
Ireland crop, so weather arriving from just outside it, mostly from the Atlantic, can't enter the
forecast. And training covered one summer only.

**Next steps:** more test dates and ensemble members, a larger region around Ireland for context,
and a loss weighted per channel to help the thermal-infrared bands.